# Exploring TC PRIMED, Chapter 1a: NetCDF Files
- Creators: Naufal Razin<sup>1</sup>, Chris Slocum<sup>2</sup>, and Kathy Haynes<sup>1</sup>
- Affiliation:
    - <sup>1</sup>Cooperative Institute for Research in the Atmosphere, Colorado State University, Fort Collins, CO
    - <sup>2</sup>NOAA/NESDIS Center for Satellite Applications and Research, Fort Collins, CO
- History:
    - August 15th, 2024: Updated for TC PRIMED v01r01
    - November 16th, 2026: Updated code to use `tcprimedapi` and new package download routine

---

## Overview
TC PRIMED uses NetCDF Groups to store data. Not all datasets stored in the NetCDF file format use NetCDF Groups. However, since TC PRIMED is a compilation of data from various sources, the NetCDF Groups functionality helps with data organization. In this notebook, you will learn how to read a TC PRIMED NetCDF file and retrieve the various information available in the different groups.

## Prerequisites
To successfully navigate and use this notebook, you should be familiar with:
- the basics of Python programming such as loading modules, assigning variables, and list/array indexing

## Learning Outcomes
By working through this notebook, you should be able to:
- understand the NetCDF file structure, particularly one that contains groups
- interact with (e.g., load and plot) data from a NetCDF file

## Time Estimate
- Estimated text reading time: 6 to 11 min
- Estimated code reading time: 3 to 5 min
- Estimated total reading time: 9 to 16 min

## Background
[NetCDF](https://www.unidata.ucar.edu/software/netcdf/), or Network Common Data Form, "is a set of software libraries and machine-independent data formats that support the creation, access, and sharing of array-oriented scientific data" (Unidata 2023). Scientific data stored in NetCDF files are meant to be self-describing in that the file should include information about the data it contains. This self-describing information is also known as attributes or metadata. NetCDF files are a type of multidimensional raster files, like GeoTIFF files.

[NetCDF Groups](https://www.unidata.ucar.edu/software/netcdf/workshops/2011/groups-types/GroupsIntro.html) are like directories or folders on your computer, except that they are contained within the NetCDF file. For example, you may have a file within a folder, and that folder is stored within a parent folder. The path to that file on your computer would be `parent_folder/child_folder/file`.

In this tutorial, you will learn how to access the attributes of the different groups in a TC PRIMED file, as well as learn how to load TC PRIMED variables within the groups.

## Software
This tutorial uses the Python programming language and packages. We will use:
- `tcprimedapi` to query and retrieve TC PRIMED files from the Amazon Web Service S3 bucket
- `netCDF4` to load the TC PRIMED file
- `numpy` for simple array operations

<div class="alert alert-block alert-danger">
    This Learning Journey assumes that the user is running the notebook in a clean conda environment. If the user runs this notebook in an existing environment, it may install a different version of an existing package, and the user would need to restart the kernel after the required package version is installed.
    <br>
    <br>
    Users should also note that support for particular packages may not be available for the newest Python versions. If the install code below produces the following error message:
    <pre><code>
        ERROR: Could not find a version that satisfies the requirement [package] (from versions: none)
        ERROR: No matching distribution found for [package]
    </code></pre>
    users will want downgrade their Python version to a version supported by <code>[package]</code>.
</div>

### Install Packages
Let's first check if we have the necessary Python packages to run this notebook. If we don't, let's install them.

In [ ]:
# sys provides access to system level information (i.e., the executable for the Python installation)
import sys

# subprocess makes system level calls
import subprocess

# json allows us to query text stored in json format
import json

# version allows us to check package version
from importlib.metadata import version

# Specify packages, their distribution name, and version needed for this notebook
PACKAGES = {"numpy": {"distribution_name": None, "version": None},
            "netCDF4": {"distribution_name": None, "version": None},
            "tcprimedapi": {"distribution_name": None, "version": None}}

restart_kernel_flag = 0

# Loop through each package to import
for package in PACKAGES:

    # Get distribution name to use for install
    dist_name = package if PACKAGES[package]["distribution_name"] is None else PACKAGES[package]["distribution_name"]
    
    try:
        
        # First, attempt to import the package
        __import__(package)

        # Then, obtain the installed package version, the required version, and the
        # latest version
        installed_version = version(dist_name)
        required_version = PACKAGES[package]["version"]

        output = subprocess.check_output([sys.executable, '-m', 'pip', 'index', 'versions', '--pre', dist_name, '--json'], text=True)
        latest_version = json.loads(output)['latest']

        # Specify the version we require based on required version or latest version
        required_version = latest_version if required_version is None else required_version

        # If the required version and doesn't match the installed version
        if installed_version != required_version:

            # Get distribution version to use for install
            dist_version = "==".join([dist_name, required_version])

            # Force reinstall the required distribution version using pip
            print('############################################################################')
            print(package)
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', dist_version])

            restart_kernel_flag = 1
            
    except (ImportError, ModuleNotFoundError):

        # Obtain the required package version
        required_version = PACKAGES[package]["version"]

        # Obtain the latest version
        output = subprocess.check_output([sys.executable, '-m', 'pip', 'index', 'versions', '--pre', dist_name, '--json'], text=True)
        latest_version = json.loads(output)['latest']

        # Specify the version we require based on required version or latest version
        required_version = latest_version if required_version is None else required_version

        # Get distribution version to use for install
        dist_version = "==".join([dist_name, required_version])

        # Install the distribution version using pip
        print('############################################################################')
        print(package)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', dist_version])

if restart_kernel_flag == 1:
    
    print('###############################################################')
    print('NEW PACKAGE VERSION INSTALLED. RESTART JUPYTER NOTEBOOK KERNEL!')
    print('###############################################################')

Now, let's load the modules in the packages (e.g., `Dataset`) or load the packages and assign a shorter object name for the packages (e.g., `import numpy as np`) for a cleaner use throughout the notebook.

In [ ]:
# Load the Python packages we will use in this notebook
import tcprimedapi
from netCDF4 import Dataset
import numpy as np

## Read File Online
Finally, let's retrieve information from the TC PRIMED file that we will use in this example. The TC PRIMED file will be from a Global Precipitation Measurement (GPM) satellite Microwave Imager (GMI) overpass of Hurricane Florence (2018). We will retrieve the TC PRIMED file from the [Amazon Web Service (AWS) S3 bucket](https://noaa-nesdis-tcprimed-pds.s3.amazonaws.com/index.html) as part of the [NOAA Open Data Dissemination program (NODD)](https://www.noaa.gov/information-technology/open-data-dissemination).

The TC PRIMED team provides a Python package to query and retrieve TC PRIMED files from the AWS S3 bucket called [`tcprimedapi`](https://pypi.org/project/tcprimedapi/). To retrieve TC PRIMED files using `tcprimedapi`, we simply provide the API with the criteria for the file(s) of interest. The criteria that are relevant to this notebook are the:
- `atcf_id`: the Automated Tropical Cyclone Forecast System (ATCF) 8 character storm identifier. The ATCF ID for Hurricane Florence (2018) is `AL062018`.
- `file_type`: which can be the sensor or platform abbreviation or the TC PRIMED environmental file. Since we are interested in looking at a GMI overpass, this will be `GMI`.

Let's query the TC PRIMED AWS S3 bucket using `tcprimedapi` and the criteria above.

In [ ]:
# Create an instance of a Client
tcpc = tcprimedapi.Client()

# List desired ATCF ID
atcf_id = ["AL062018"]

# List desired file type
file_type = ["GMI"]

# Setup a data request dictionary
data_request = {"atcf_id": atcf_id,
                "file_type": file_type}

# Make a data request query
tcpc.query(data_request)

As you can see, the query returned 17 files that match our criteria. Let's list all of them.

In [ ]:
for obj_key in tcpc.object_keys:
    print(obj_key)

Let's retrieve and explore data from `TCPRIMED_v01r01-final_AL062018_GMI_GPM_025677_20180905051440.nc`.

Note that `tcprimedapi` is designed to safely handle a large volume of data. The conventional way of using `tcprimedapi` involves iterating over the available files from the query and individually downloading each file or reading into memory specific variables from each file while ensuring that each file is properly closed.

However, for this notebook, we will be retrieving and exploring data from a single file, which requires the file to remain open until we are done with it. So, we will use the `iter()` and `next()` functions.

Therefore, we will take the following steps:
- Redefine `object_keys` to have just the file of interest (`TCPRIMED_v01r01-final_AL062018_GMI_GPM_025677_20180905051440.nc`).
- Retrieve the datastream object associated with the file.
- Load the datastream object into a NetCDF instance called "DS" without downloading the file. The data will exist only in your local system's random access memory (RAM).

In [ ]:
# Redefine object_keys with just the file of interest
new_key = "v01r01/final/2018/AL/06/TCPRIMED_v01r01-final_AL062018_GMI_GPM_025677_20180905051440.nc"
tcpc.object_keys = [new_key]

# Retrieve the datastream associated with the file
obj_key, obj_memory = next(iter(tcpc.inmemory()))

# Load the datastream object into a NetCDF instance
DS = Dataset(obj_key, memory=obj_memory)

Great! We've loaded the data into a NetCDF "instance". Let's first look at the instance of the file at the "root" group.

The root group is the outer-most "directory" or "folder" in a NetCDF file. Its instance gets automatically loaded when you load a NetCDF file in Python. For datasets that do not use NetCDF Groups, all variables would be stored in the root group. However, in TC PRIMED, the root group stores only the "global" attributes &mdash; information about the whole file &mdash; and the different sub-groups.

Let's print the root group instance.

In [ ]:
# DS would automatically contain information from the root group
# Print the instance of the file from the root group
print(DS)

The printout for the root group above shows all of the global attributes &mdash; e.g., dataset title, file ID, product version, etc. You can simply glean those information from the output above.

Having gleaned the global attributes above, you can directly load a particular attribute from the root group using the `getncattr` function.

In [ ]:
# Load the "summary" attribute from the root group as root_summary
root_summary = DS.getncattr("summary")

# Print root_summary
print(root_summary)

<div class="alert alert-block alert-success">
<h3>Exercise 1</h3>
Using the example above, uncomment the template below, and change the code as necessary to load the "title" attribute from the root group.
</div>

<div class="alert alert-block alert-info">
<b>Hint:</b> If you are unsure about what attributes are available, print out the root group instance again.
</div>

In [ ]:
# Load the "title" attribute from the root group as root_title
#root_title = DS.getncattr("insert_attr_here")

# Print root_title
#print(root_title)

Awesome! Now, let's step back and look at the root group again, printed out below

In [ ]:
# Print the instance of the file from the root group
print(DS)

Notice at the end of the printout, the available "dimensions," "variables," and "groups" are listed for the root group. **These are not part of the TC PRIMED file metadata, but are part of the standard output of the Python netCDF4 package as additional information**. Nonetheless, we can use that information. Under groups, six groups are listed. They are:
- `overpass_metadata`
- `overpass_storm_metadata`
- `passive_microwave`
- `GPROF`
- `radar_radiometer`
- `infrared`

<div class="alert alert-block alert-warning">
<b>Be careful.</b> The <code>radar_radiometer</code> group is only available for satellites with precipitation radars, such as the TRMM and GPM. Other groups are available across the different sensors and satellites in TC PRIMED.
</div>

Now, let's access the instance of the `passive_microwave` group.

In [ ]:
# Using the file instance, load the passive_microwave group as a
# group instance called passive_microwave_group
passive_microwave_group = DS["passive_microwave"]

# Print the passive_microwave group instance
print(passive_microwave_group)

Notice, as in the root group, the printout above indicates the dimensions, variables, and groups that are available in the `passive_microwave` group. In this example, you have
- a `time` dimension of length 1
- a `time` variable with a `time` dimension; therefore, a `time` variable of length 1
- two sub-groups: `S1` and `S2`

Let's load and print out the `time` variable instance.

In [ ]:
# Using the file instance, load the variable instance for time
# in the passive_microwave group by specifying its "path" in the file
pm_time_instance = DS["passive_microwave/time"]

# Print the variable instance of passive microwave time
print(pm_time_instance)

From the printout of the time variable instance, you can see information on the `time` variable. As you have seen from printing out the passive microwave group instance above, `time` is a one-dimensional array with one element (`current shape = (1,)`). Let's first retrieve the `long_name` attribute of this time instance using a function we've already used above, `getncattr`.

In [ ]:
# Using the file instance, load the variable instance for time
# in the passive_microwave group by specifying its "path" in the file
# Then, retrieve the long_name attribute using getncattr
pm_time_long_name = DS["passive_microwave/time"].getncattr("long_name")

# Print the time variable long_name
print(pm_time_long_name)

Finally, let's retrieve the `time` variable itself from the `time` variable instance. To do so, load the variable from the `time` variable instance using `[:]`

In [ ]:
# Using the file instance, load the time variable from the
# passive_microwave group by specifying its "path" in the file and
# using [:] to load the variable
pm_time = DS["passive_microwave/time"][:]

# Print the passive microwave time variable
print(pm_time)

You've loaded a NetCDF variable! As you have seen in the cells above, it is a one-dimensional array with one element. The `time` variable in TC PRIMED files are in the units of seconds since 1970-01-01T00:00:00. We will briefly discuss this `time` unit in Chapter 1c.

For now, let's go back to the `passive_microwave` group instance.

In [ ]:
# Using the file instance, load the passive_microwave group as a
# group instance called passive_microwave_group
passive_microwave_group = DS["passive_microwave"]

# Print the passive_microwave group instance
print(passive_microwave_group)

<div class="alert alert-block alert-success">
<h3>Exercise 2</h3> Notice that within the <code>passive_microwave</code> group, there are two additional groups called <code>S1</code> and <code>S2</code>. We will discuss these groups in more detail in Chapter 1b. But for now, using the knowledge you've obtained about NetCDF groups, instances, and variables, uncomment the code below and change the code as necessary to print out the <code>S1</code> group instance from the <code>passive_microwave</code> group.

In [ ]:
# Using the file instance, load the S1 group within the passive_microwave
# group as an instance called S1_group
#S1_group = DS["insert_S1_group_here"]

# Print the S1 group instance
#print(S1_group)

From the printout of the `S1` group instance above, you should see two dimensions
- `scan` with length 171
- `pixel` with length 221

You should also see a list of variables such as `latitude` and `longitude`, all of which should have a `scan` dimension or both `scan` and `pixel` dimension.

<div class="alert alert-block alert-success">
<h3>Exercise 3</h3> Using the knowledge you've obtained about NetCDF groups, instances, and variables, uncomment the template below and change the code as necessary to load and print out the <code>latitude</code> variable.
</div>

<div class="alert alert-block alert-info">
<b>Hint:</b> When loading variables, don't forget to use <code>[:]</code>

In [ ]:
# Using the file instance, load the latitude variable from the
# passive_microwave group and S1 sub-group by specifying its "path"
# in the file, and using [:] to load the variable
#S1_latitude = DS["insert_latitude_var_here"]

# Print the latitude variable from the passive_microwave group and
# the S1 sub-group
#print(S1_latitude)

When loading NetCDF variables, the `[:]` operator loads the full array of the variable. But, what if you only wanted the first 10 entries of the `scan` dimension? You can refer to the dimension information for each variable by first printing out the group or variable instance, like you've done in the cells above. Then, simply supply the read routine with the appropriate index. Let's look at an example by printing out the size of the `latitude` variable being loaded.

In [ ]:
# Load the full array of the latitude variable using only [:]
S1_full_latitude = DS["passive_microwave/S1/latitude"][:]

# Print the shape of the full latitude array
print(S1_full_latitude.shape)

# Load a subset of the latitude variable for the first 10 entries of the
# scan dimension and all of the pixel dimension
S1_subset_latitude = DS["passive_microwave/S1/latitude"][np.arange(0,10,1),:]

# Print the shape of the latitude variable subset
print(S1_subset_latitude.shape)

<div class="alert alert-block alert-success">
<h3>Exercise 4</h3> Uncomment the template below and change the code as necessary to load the first 20 entries of the <code>scan</code> dimension and the first 10 entries of the <code>pixel</code> dimension for the latitude variable. Then, print out the shape of the variable array.

In [ ]:
# Load a subset of the latitude variable for the first 20 entries of the
# scan dimension and the first 10 entries of the pixel dimension
#S1_subset_latitude = DS["insert_latitude_var_here"]

# Print the shape of the latitude variable subset
#print(S1_subset_latitude.shape)

## Close the File
When loading data from a NetCDF file, **always remember to close the file**. A best practice would be to close the file immediately after loading the variable or attribute of interest. However, since we're loading various variables and attributes throughout this notebook, we will close the file at the end of this notebook using the command below.

In [ ]:
DS.close()

## Final Thoughts
There are other ways to query NetCDF files, like using [Unidata's NetCDF Utilities](https://docs.unidata.ucar.edu/nug/current/netcdf_utilities_guide.html). However, in this tutorial, you learned how to use the Python `netCDF4` package to:
- retrieve information from a remote NetCDF file
- print out NetCDF instances
- obtain attributes from NetCDF instances
- retrieve a variable from NetCDF instances
- navigate groups in a NetCDF file
- load only a subset of data from a NetCDF variable

These are crucial steps for you to get familiarized with the structure of TC PRIMED files before you move on to conducting analyses on TC PRIMED data.

## Data Statement
- Razin, Muhammad Naufal; Slocum, Christopher J.; Knaff, John A.; Brown, Paula J. 2023. Tropical Cyclone PRecipitation, Infrared, Microwave, and Environmental Dataset (TC PRIMED). v01r01. NOAA National Centers for Environmental Information. https://doi.org/10.25921/dmy1-0595.

## References
- Unidata, 2023: Network Common Data Form (NetCDF). Accessed 13 June 2023, https://www.unidata.ucar.edu/software/netcdf/.

## Metadata
- Language / package
    - Python
    - tcprimedapi
    - netCDF4
    - numpy
- Application keywords
    - NetCDF
    - NetCDF Groups
- Geophysical keywords
    - Tropical Cyclones

## Disclaimer

This Jupyter notebook is a scientific product and is not official communication of the National Oceanic and Atmospheric Administration, or the United States Department of Commerce. All NOAA Jupyter notebooks are provided on an 'as is' basis and the user assumes responsibility for its use. Any claims against the Department of Commerce or Department of Commerce bureaus stemming from the use of this Jupyter notebook will be governed by all applicable Federal law. Any reference to specific commercial products, processes, or services by service mark, trademark, manufacturer, or otherwise does not constitute or imply their endorsement, recommendation or favoring by the Department of Commerce. The Department of Commerce seal and logo, or the seal and logo of a DOC bureau, shall not be used in any manner to imply endorsement of any commercial product or activity by DOC or the United States Government.